In [1]:
import pandas as pd
from datetime import datetime as dt

from rockyclickup.wrapper import Session as rcu_session
from rockyclickup.models import FSA, LSA, DCA, HRA, HSA, TRN, PKG, EDU, ADO, Client
from rockyclickup.utils import response_to_dataframe as rcu_res_to_df, datetime_nearest_day

from rockyelevate.wrapper import Session as elv_session
from rockyelevate.utils import response_to_dataframe

clickup = rcu_session()
elv = elv_session('PROD', multithread=True, max_threads=200)

In [2]:
# ~ 9 mins @ 200 threads
org_cache_path = "all_orgs_250903.pkl"

try:
    all_orgs_df = pd.read_pickle(org_cache_path)
except FileNotFoundError as e:
    all_orgs = elv.get_organizations(
        statuses=["PENDING", "ACTIVE", "TERMINATED", "ACTIVATION_FAILED"],
        types=["SYSTEM", "PARTNER", "DISTRIBUTOR", "COMPANY", "SUBSIDIARY", "SUBGROUP"],
        subsidiaries='include'
    )
    all_orgs_df = response_to_dataframe(all_orgs)
    all_orgs_df.to_pickle(org_cache_path)

org_ids = all_orgs_df['id'].unique()

In [ ]:
# ~ 15mins @ 200 threads
plan_cache_path = "all_plans_250903.pkl"

try:
    all_plans_df = pd.read_pickle(plan_cache_path)
except FileNotFoundError as e:
    all_plans = elv.get_plans_by_org(
        oids=org_ids,
        detail=True
    )
    all_plans_df = response_to_dataframe(all_plans)
    all_plans_df.to_pickle(plan_cache_path)

In [ ]:
for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    all_plans_df[col] = pd.to_datetime(all_plans_df[col])
    all_plans_df[col] = all_plans_df[col].fillna(dt(2000, 1, 1))

In [ ]:
j_plans = all_plans_df[
    (all_plans_df['plan_year.valid_from'] >= dt(2025, 6, 30)) &
    (all_plans_df['plan_year.valid_from'] <= dt(2025, 7, 2))
]
print(len(j_plans))

In [ ]:
all_plan_responses = []

for model in [FSA, LSA, DCA, HRA, HSA, TRN, PKG, EDU, ADO]:
    full_list = clickup.get_full_list(model=model)

    all_plan_responses.extend(full_list)

all_clients = clickup.get_full_list(model=Client)
all_clients_df = rcu_res_to_df(all_clients)
all_clients_df = all_clients_df.rename(columns={'id': 'client_id'})

In [ ]:
all_clickup_plans_df = rcu_res_to_df(all_plan_responses)

def assign_client_id(filtered_list):
    unique = list(set(filtered_list))

    if len(unique) == 0:
        return None
    
    if len(unique) == 1:
        return unique[0]
    
    print("none found")
    return None


relation_fields = [c for c in all_clickup_plans_df if 'client' in c]
all_clickup_plans_df[relation_fields] = all_clickup_plans_df[relation_fields].apply(
    lambda col: col.apply(lambda x: x if isinstance(x, list) else [])
)
all_clickup_plans_df['client_id_list'] = all_clickup_plans_df.apply(lambda row: sum(row[relation_fields].values, []), axis=1)
all_clickup_plans_df['client_id_list'] = all_clickup_plans_df['client_id_list'].apply(lambda x: list(set(x)))
all_clickup_plans_df['client_id'] = all_clickup_plans_df['client_id_list'].apply(assign_client_id)

filtered_clients = all_clients_df[all_clients_df['client_id'].isin(all_clickup_plans_df['client_id'])]
client_plan_df = pd.merge(left=all_clickup_plans_df, right=all_clients_df, on='client_id')
for col in ['date_plan_start', 'date_plan_end']:
    client_plan_df[col] = client_plan_df[col].fillna(dt(2000, 1, 1))
    client_plan_df[col] = client_plan_df[col].apply(lambda x: datetime_nearest_day(x))

client_plan_df['key'] = client_plan_df.apply(lambda row: f"{row['rmr_code']} | {row['list.name_x']} | {dt.strftime(row['date_plan_start'], "%m%d%y")} | {dt.strftime(row['date_plan_end'], "%m%d%y")}", axis=1)

In [ ]:
account_type_map = {
    "HRA":              "HRA",
    "HSA":              "HSA",
    "HCFSA":            "FSA",
    "DCAP":             "DCA",
    "TRANSIT":          "TRN",
    "PARKING":          "PKG",
    "LIFESTYLE":        "LSA"
}

external_identifier_map = {
    org['id']: org['external_identifier']
    for index, org in all_orgs_df.iterrows()
}

all_plans_df['external_identifier'] = all_plans_df['organization_id'].map(external_identifier_map)

all_plans_df['account_type'] = all_plans_df['account_type.account_type'].apply(lambda x: account_type_map.get(x))

all_plans_df['key'] = all_plans_df.apply(lambda row: f"{row['external_identifier']} | {row['account_type']} | {dt.strftime(row['plan_year.valid_from'], "%m%d%y")} | {dt.strftime(row['plan_year.valid_to'], "%m%d%y")}", axis=1)


In [ ]:
merge_df = pd.merge(left=all_plans_df, right=client_plan_df, on='key', how='left')


In [ ]:
july_plans = merge_df[
    (merge_df['plan_year.valid_from'] >= dt(2025, 6, 30)) & 
    (merge_df['plan_year.valid_from'] <= dt(2025, 7, 2))
]

In [ ]:
july_plans = july_plans.rename(columns={
    "id_x": "elv_plan_id",
    "id_y": "clickup_plan_id",
    'plan_account_funding_config.max_rollover_amount.max_rollover_amount': "max_rollover_amount (elv)",
    'plan_account_funding_config.is_rollover.is_rollover': 'is_rollover',
    'rollover_max': 'rollover_max (clickup)'
})

# display(july_plans)

cols_to_show = ['elv_plan_id', 'plan_code', 'is_rollover', 'max_rollover_amount (elv)', 'rollover_max (clickup)', 'clickup_plan_id']

for col in cols_to_show:
    july_plans[col] = july_plans[col].fillna("")

# display(july_plans[cols_to_show])

In [ ]:
df_to_export = july_plans[cols_to_show].drop_duplicates().sort_values(by=['is_rollover', 'plan_code'], ascending=False)
pd.set_option('display.max_rows', None)
df_to_export.to_csv("july_rollover_max_audit.csv")
# print(len(df_to_export))
# display(df_to_export.sort_values('creator.username_x'))

In [ ]:
filtered_df = df_to_export[
    (df_to_export['max_rollover_amount (elv)'] != "") |
    (df_to_export['rollover_max (clickup)'] != "")
]


filtered_df